# Python Implementation of the HO-CAE Algorithm
This notebook contains the python implementation of the hardware optimized, cell averaging estimation algorithm as detailed [here](https://ieeexplore.ieee.org/document/10058972)

## Pre-Processing
Import the data, validate, and plot a section of it to ensure that there is enough 

In [ ]:
from utils.data import load_iq_data

FILE_NAME = "usrp_2.45g_test_capture.bin"

# Load the IQ data from the binary file
data = load_iq_data(FILE_NAME)
print(f"Loaded {len(data)} samples from {FILE_NAME}")

In [ ]:
# Get all of the constants of the data. 
# In the future we should store this as metadata alongside the .bin file, but for now we will hardcode it here.
SAMP_RATE = 100e6
CENTER_FREQ = 2.45e9
DURATION = len(data) / SAMP_RATE


In [ ]:
from utils.plot import plot_spectrogram

# plot a section of the data to understand what it looks like before the algorithm
data_to_plot = data[:int(SAMP_RATE * 0.01)] # use the first 10ms of the IQ data for plotting
plot_spectrogram(data_to_plot, SAMP_RATE, CENTER_FREQ)

In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

# Ensure IQ data is a one-dimensional complex array
data = np.asarray(data[:int(SAMP_RATE*0.01)]).squeeze()

if data.ndim != 1:
    raise ValueError(f"Expected 1D IQ data, received shape {data.shape}")

# STFT parameters
NFFT = 1024
OVERLAP = 512

# Compute the short-time Fourier transform
f, t, z = signal.stft(
    data,
    fs=SAMP_RATE,
    window="hann",
    nperseg=NFFT,
    noverlap=OVERLAP,
    nfft=NFFT,
    return_onesided=False,
    boundary=None,
    padded=False,
)

# Put negative frequencies on the left and positive frequencies on the right
f_shifted = np.fft.fftshift(f)
z_shifted = np.fft.fftshift(z, axes=0)

magnitude_sq = np.abs(z_shifted) ** 2
magnitude_sq = magnitude_sq.T

magnitude_db = 20.0 * np.log10(np.maximum(np.abs(z_shifted), 1e-12))
magnitude_db = magnitude_db.T

print(f"magnitude_sq max value: {np.max(magnitude_sq)}, min value: {np.min(magnitude_sq)}, mean value: {np.mean(magnitude_sq)}")

## HO-CAE Implementation 
The following is the implementation of the HO-CAE algoirthm in python. In this example, we will apply the algorithm, and then we will use the tresholds to apply a mask to the signal, and make sure that there is no noise leakage while perserving all of the spectral properties we need to apply the FSS algoirthm to. You can play with the n, k, and alpha properties of the algoirthm to see how it affects the threshold (and thus the mask). The paper this algoirthm is based on asserts that the best universal values for n, k, and alpha are 64, 5, and 16 respectively. 


In [ ]:
# define the constants
window_size = 64 # n
order_statistic = 5 # k
alpha = 13

In [ ]:
thresholds = []

for example_bin in magnitude_sq:
    # example_bin = magnitude_db[0]
    p = 0
    est = []
    while p < len(example_bin):
        est.append(np.average(example_bin[p:p+window_size]))
        p += int(window_size//2)

    if len(est) != ((len(example_bin) * 2) // window_size):
        raise ValueError(f"Expected {((len(example_bin) * 2) // window_size)} estimates, got {len(est)}")

    est.sort(reverse=False)
    thresholds.append(est[order_statistic] * alpha)

In [ ]:
# in order to double check the algorithm, we can use it to apply a mask to the magnitude_db and see if it removes the noise and leave the signal
masked_data = np.copy(magnitude_sq)
for i, threshold in enumerate(thresholds):
    masked_data[i][masked_data[i] < threshold] = 0
    masked_data[i][masked_data[i] >= threshold] = 1


plt.figure(figsize=(12, 6))
plt.pcolormesh(f_shifted, t, masked_data, shading='gouraud', cmap='viridis')
plt.ylabel("Time [s]")
plt.xlabel("Frequency [Hz]")
plt.title("Masked Spectrogram")
plt.colorbar(label='Masked Data')
plt.show()